In [1]:
import pandas as pd

class PDBparser:
    
    def __init__(self, pdbPath):
        self.__pdbPath = pdbPath

    def get_residues(self):
        with open(self.__pdbPath, 'r') as file:
            aa_map = {
                'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D',
                'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G',
                'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
                'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S',
                'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'}
            
            seen_residues = {}

            for line in file:
                line = line.strip()

                if line.startswith('ATOM'):
                    res_name = line[17:20]
                    chain = line[21]
                    res_num = int(line[22:26])

                    if chain not in seen_residues:
                        seen_residues[chain] = {}

                    if res_num not in seen_residues[chain]:
                        seen_residues[chain][res_num] = aa_map.get(res_name, 'err')
            
            sequences = {}
            for chain, residues in seen_residues.items():
                sequences[chain] = "".join(residues[num] for num in sorted(residues))     

        df = pd.DataFrame(
            list(sequences.items()),
            columns=['chain', 'sequence']
        )

        return df


In [11]:

pdb = PDBparser("1CY1.pdb")
pdb_parsed = pdb.get_residues()
pdb_parsed.head()

,chain,sequence
0,A,GKALVIVESPAKAKTINKYLGSDYVVKSSVGHIRDLPRGALVNRMG...


In [9]:
pdb_parsed.head()

pdb_parsed = pdb_parsed.reset_index(drop=True)              # ensure stable index
pdb_parsed['seq_list'] = pdb_parsed['sequence'].apply(list) # turn sequence into list of chars

pdb_parsed_residues = (
    pdb_parsed[['chain', 'seq_list']]
    .explode('seq_list')
    .rename(columns={'seq_list': 'residue'})
    .reset_index(drop=True)
)[['chain', 'residue']]
pdb_parsed_residues.head()

,chain,residue
0,A,G
1,A,K
2,A,A
3,A,L
4,A,V
